# pytempo: un tur ghidat

`pytempo` citeste statistica oficiala romaneasca din API-ul INS TEMPO Online:
gaseste indicatori, le citeste metadatele si aduce datele intr-un DataFrame
pandas.

**Acest notebook ruleaza live pe serverul INS.** Fiecare celula face cereri
reale, deci are nevoie de conexiune, iar cateva celule dureaza cateva
secunde. Exemplele au fost alese sa ramana mici si politicoase: nicio celula
de aici nu face mai mult de cateva cereri.

Se instaleaza direct din GitHub:

    pip install git+https://github.com/CIDS-UBB/pytempo.git


In [1]:
import pytempo as t

## 1. Descoperire

Incepe cu o panorama: cat de mare e catalogul si de unde se porneste.


In [2]:
t.overview()

pytempo: 1916 TEMPO indicators, in 8 top level domains.
Start with find('salariati') or domains(). t.help() has the full guide.


`find` e cautarea simpla pe cuvinte. Se uita in numele indicatorului si in
cod, ignora diacriticele si majusculele, si cere ca **toate** cuvintele date
sa se potriveasca. Intoarce **toate** potrivirile, nu o pagina taiata, deci
taie rezultatul daca vrei doar cateva.


In [3]:
rezultate = t.find("salariati")
print(len(rezultate), "indicators")
rezultate[:5]

104 indicators


[Matrix('AMG1103', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si sexe'),
 Matrix('AMG1104', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si medii de rezidenta'),
 Matrix('AMG1105', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si sexe'),
 Matrix('AMG1106', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si medii de rezidenta'),
 Matrix('AMG115K', 'AMIGO - Salariati cu regim de lucru temporar dupa durata obisnuita a saptamanii de lucru si sexe')]

`search` e cealalta unealta: descoperire cu filtre. Cuvantul cautat e
optional, iar filtrele se combina intre ele.

* `domeniu` se potriveste pe un subsir din numele domeniului statistic, deci
  `economic` gaseste `B. STATISTICA ECONOMICA` fara sa stii forma exacta.
* `periodicitate` se potriveste pe un subsir din cat de des se publica.
* `level` pastreaza doar indicatorii care ajung la acel nivel teritorial.
* `caen=True` pastreaza doar pe cei cu o clasificare de activitati CAEN.

Diferenta intr-o linie: **`find` cauta in nume, `search` filtreaza pe
metadate.**


In [4]:
economici = t.search(domeniu="economic", periodicitate="anuala", level="judet")
print(len(economici), "indicators")
economici[:5]

111 indicators


[Matrix('AGR101A', 'Suprafata fondului funciar dupa modul de folosinta, pe forme de proprietate, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR101B', 'Suprafata fondului funciar dupa modul de folosinta, pe judete si localitati'),
 Matrix('AGR102A', 'Suprafata terenurilor amenajate cu lucrari de irigatii si suprafata agricola irigata, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102B', 'Suprafata terenurilor amenajate cu lucrari de desecare, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102C', 'Suprafata terenurilor amenajate cu lucrari de ameliorare si combaterea eroziunii solului, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete')]

`t.filters()` afiseaza pe ce se poate filtra, cu valorile reale citite din
catalog, nu dintr-o lista scrisa de mana.


In [5]:
t.filters()

Filters for t.search(). They combine with each other and with the search words.
  level        : ['national', 'macroregiune', 'regiune', 'judet', 'localitate', 'necunoscut']
  caen         : True only those with a CAEN dimension, False only those without
  domeniu      : substring of the domain name, diacritics ignored
                 A. STATISTICA SOCIALA
                 B. STATISTICA ECONOMICA
                 C. FINANTE
                 D. JUSTITIE
                 E. MEDIU INCONJURATOR
                 F. UTILITATI PUBLICE SI ADMINISTRAREA TERITORIULUI
                 G. DEZVOLTARE DURABILA - Orizont 2020
                 H. DEZVOLTARE DURABILA - Tinte 2030
  periodicitate: substring of the periodicity, diacritics ignored
                 ['5 - 6 ani', 'Anuala', 'Cincinal', 'La 2 ani', 'La fiecare 3 ani sau mai mult', 'La trei ani', 'Lunara', 'Perioada neregulata', 'Recomandabil la 10 ani', 'Trimestriala']

The metadata filters rest on the local index. If it is missing,
search a

## 2. Intelegerea unui indicator

Inainte sa tragi date, citeste ce este de fapt indicatorul. Folosim FOM104D,
numarul mediu al salariatilor pe judete si localitati.


In [6]:
m = t.matrix("FOM104D")
m

Matrix('FOM104D', 'Numarul mediu al salariatilor pe judete si localitati')

`what()` e versiunea scurta: prima fraza din definitie, unitatea de masura,
cat de des se publica, cand a fost actualizat ultima data, si un avertisment
daca observatiile mentioneaza ani anume, ceea ce de obicei semnaleaza o
ruptura de serie.


In [7]:
m.what()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
  Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
  unit        : Numar persoane
  periodicity : Anuala
  updated     : 20-11-2025
                read them with .describe()


`where()` arata unde sta indicatorul in arborele de domenii si ce acopera:
cate unitati teritoriale pe fiecare nivel, daca localitatile poarta cod
SIRUTA, si intervalul de ani.

O nota despre **nivelele teritoriale**. `national`, `macroregiune`,
`regiune`, `judet`, `localitate` sunt felul in care pytempo interpreteaza
denumirile optiunilor, nu un concept expus direct de INS. O dimensiune
teritoriala amesteca de obicei toate nivelele intr-o singura coloana, iar
pytempo deduce care e care, ca sa poti cere unul anume. Denumirile care nu
se incadreaza in nomenclatorul administrativ, cum sunt statiile de
monitorizare, primesc `necunoscut` in loc sa fie fortate intr-un nivel.


In [8]:
m.where()

domain   : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
territory: Judete (43 options)
    national        1
    judet           42
territory: Localitati (3183 options)
    localitate      3183
SIRUTA   : yes
time     : Ani, 35 periods, 1990 to 2024


`how()` genereaza manualul de descarcare pentru acest indicator anume:
comenzile care au sens pentru el, strategia care se va folosi, si la cate
cereri sa te astepti.


In [9]:
m.how()

How to download FOM104D:
  m = t.matrix('FOM104D')
  df = m.get()          level localitate, tidied
  (the level filter does not apply here: county and locality are
   separate dimensions, and get() brings both anyway)
  m.get(raw=True)       exactly what INS returns, no extras

  strategy: by_county, roughly 43 requests
  downloaded in several requests and concatenated


`describe()` afiseaza fisa integrala, exact cum a scris-o INS: definitia
completa, metodologia, sursele si observatiile. E lunga intentionat.
Observatiile sunt locul unde stau rupturile de serie si avertismentele
despre ani incompleti, deci citeste-le inainte sa te bazezi pe o serie.


In [10]:
m.describe()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
domain      : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
levels      : national, judet, localitate
periodicity : Anuala
updated     : 20-11-2025

DEFINITION
Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
Numarul mediu al salariatilor se calculeaza ca medie aritmetica simpla rezultata din suma efectivelor zilnice de salariati (exclusiv cei al caror contract de munca/raport de serviciu a fost suspendat), din perioada de referinta, inclusiv din zilele de repaus saptamanal, sarbatori legale si alte zile nelucratoare, impartita la numarul total al zilelor calendaristice.
In efectivul zilnic al salariatilor luat in calculul numarului mediu se cuprind urmatoarele categorii:
- sal

`options()` fara argument listeaza dimensiunile, fiecare cu rolul pe care
i l-a dat pytempo si cu cate valori are.


In [11]:
m.options()

[0] Judete (teritoriu, 43 options)
[1] Localitati (teritoriu, 3183 options)
[2] Ani (timp, 35 options)
[3] UM: Numar persoane (um, 1 options)

Cu un argument listeaza valorile unei dimensiuni. O poti numi dupa label,
dupa rol, dupa index sau dupa nivel.


In [12]:
m.options("teritoriu", limit=8)

TOTAL, 1017 MUNICIPIUL ALBA IULIA, 1213 MUNICIPIUL AIUD, 1348 MUNICIPIUL BLAJ, 1874 MUNICIPIUL SEBES, 1151 ORAS ABRUD, 2915 ORAS BAIA DE ARIES, 1455 ORAS CAMPENI

## 3. O extragere simpla

FOM101A, resursele de munca pe judete, incape intr-o singura cerere, deci e
o prima tragere buna.

`get()` fara argumente face trei lucruri implicit: alege cel mai fin nivel
teritorial pe care il atinge indicatorul, aplica standardizarea tidy, si
afiseaza o linie cu ce a hotarat.


In [13]:
df = t.matrix("FOM101A").get()
df.shape

FOM101A: level judet (finest), single, 1 request


(4392, 10)

Rezultatul e in **format lung**: un rand per combinatie, o coloana text per
dimensiune, o coloana numerica `Valoare`, si apoi coloanele derivate adaugate
de tidy.


In [14]:
df.head()

,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare,"Macroregiuni, regiuni de dezvoltare si judete_siruta","Macroregiuni, regiuni de dezvoltare si judete_nivel","Macroregiuni, regiuni de dezvoltare si judete_tip","Macroregiuni, regiuni de dezvoltare si judete_nume",Ani_an
0,Total,Arges,Anul 1990,Mii persoane,394.8,<NA>,judet,<NA>,Arges,1990
1,Total,Arges,Anul 1991,Mii persoane,394.7,<NA>,judet,<NA>,Arges,1991
2,Total,Arges,Anul 1992,Mii persoane,399.5,<NA>,judet,<NA>,Arges,1992
3,Total,Arges,Anul 1993,Mii persoane,392.9,<NA>,judet,<NA>,Arges,1993
4,Total,Arges,Anul 1994,Mii persoane,392.2,<NA>,judet,<NA>,Arges,1994


Coloanele derivate sunt tipizate, nu text: `Int64` pentru codul SIRUTA si
pentru an, string nullable pentru nivel, tip si denumirea curata.


In [15]:
df.dtypes

Sexe                                                        str
Macroregiuni, regiuni de dezvoltare si judete               str
Ani                                                         str
UM: Mii persoane                                            str
Valoare                                                 float64
Macroregiuni, regiuni de dezvoltare si judete_siruta      Int64
Macroregiuni, regiuni de dezvoltare si judete_nivel      string
Macroregiuni, regiuni de dezvoltare si judete_tip        string
Macroregiuni, regiuni de dezvoltare si judete_nume       string
Ani_an                                                    Int64
dtype: object

Cererea unui nivel schimba ce vine inapoi. Judetele si regiunile sunt felii
diferite din acelasi indicator, deci numarul de randuri difera.


In [16]:
judete = t.matrix("FOM101A").get(level="judet", progress=False)
regiuni = t.matrix("FOM101A").get(level="regiune", progress=False)
print("judet  :", judete.shape)
print("regiune:", regiuni.shape)

judet  : (4392, 10)
regiune: (840, 10)


`raw=True` da exact ce a intors INS, fara coloane derivate. Foloseste brut
cand vrei sa vezi sursa neatinsa sau iti scrii propria prelucrare; foloseste
tidy, care e implicit, cand vrei sa lucrezi cu datele.


In [17]:
brut = t.matrix("FOM101A").get(raw=True, progress=False)
print("raw :", brut.shape)
print("tidy:", df.shape)
brut.head(3)

raw : (4392, 5)
tidy: (4392, 10)


,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare
0,Total,Arges,Anul 1990,Mii persoane,394.8
1,Total,Arges,Anul 1991,Mii persoane,394.7
2,Total,Arges,Anul 1992,Mii persoane,399.5


## 4. Standardizarea si SIRUTA

SIRUTA e codul oficial al unei unitati administrative din Romania. INS il
pune in interiorul denumirii localitatii, ca prefix numeric, deci eticheta
bruta arata asa: `1017 MUNICIPIUL ALBA IULIA`.

pytempo il desface in coloane separate si **pastreaza denumirea originala
neatinsa**. SIRUTA se pastreaza ca si CHEIE, niciodata nu se arunca, fiindca
el e cel care iti permite sa legi datele astea de alte surse administrative:
registre de populatie, bugete, geografii.

Folosim aici SAN103B, copiii inscrisi in crese pe judete si localitati,
fiindca ajunge la nivel de localitate si totusi incape intr-o singura cerere.


In [18]:
crese = t.matrix("SAN103B").get()
localitati = crese[crese["Localitati_nivel"] == "localitate"]
localitati[["Localitati", "Localitati_siruta", "Localitati_tip",
            "Localitati_nume", "Valoare"]].head(6)

SAN103B: all levels, single, 1 request


,Localitati,Localitati_siruta,Localitati_tip,Localitati_nume,Valoare
2,1017 MUNICIPIUL ALBA IULIA,1017,municipiu,ALBA IULIA,50
3,1213 MUNICIPIUL AIUD,1213,municipiu,AIUD,41
5,9262 MUNICIPIUL ARAD,9262,municipiu,ARAD,159
6,9459 ORAS CHISINEU-CRIS,9459,oras,CHISINEU-CRIS,14
7,9538 ORAS INEU,9538,oras,INEU,18
8,9574 ORAS LIPOVA,9574,oras,LIPOVA,50


Observa ce s-a intamplat: codul, tipul de asezare si denumirea curata sunt
acum trei coloane separate si tipizate, in timp ce coloana `Localitati`
pastreaza textul original.

Inca un lucru de stiut: **datele sunt rare.** Combinatiile fara date lipsesc
ca randuri intregi, nu vin ca `NaN`. Asta reflecta istorie administrativa
reala, nu o lipsa a bibliotecii: Ilfov si Municipiul Bucuresti nu exista ca
unitati separate inainte de 1996, deci acele randuri pur si simplu nu exista.
Nu verifica o descarcare comparand numarul de randuri cu produsul
dimensiunilor.


In [19]:
print("rows returned :", len(crese))
print("empty values  :", int(crese["Valoare"].isna().sum()))

rows returned : 180
empty values  : 0


## 5. O descarcare mai mare, cu spargere automata

**Aceasta celula dureaza mai mult decat celelalte.** Face mai multe cereri.

Un singur POST catre INS e limitat la un buget de celule. Cand un indicator e
mai mare de atat, pytempo sparge lucrul automat si concateneaza bucatile:
judet cu judet pentru indicatorii care ajung la nivel de localitate, altfel
pe cea mai mare dimensiune. Nu trebuie sa planifici nimic, dar e bine sa stii
ca se intampla, si de aceea `get()` afiseaza linia de decizie.

FOM106E e o demonstratie buna: se sparge pe dimensiunea CAEN in cateva
cereri. Indicatorii care ar cere sute de cereri nu sunt folositi in acest
tutorial, iar `get()` cere confirmare inainte sa porneasca unul.


In [20]:
big = t.matrix("FOM106E").get()
big.shape

FOM106E: level judet (finest), split:CAEN Rev.2  (activitati ale economiei nationale), 2 requests


  1/2: +84354 rows (total 84354)


  2/2: +45256 rows (total 129610)


(129610, 11)

## 6. O mica analiza

Datele vin gata de folosit. Nimic de mai jos nu tine de pytempo: de aici
incolo e pandas obisnuit.

Luam cadrul tidy FOM101A, pastram un judet si citim seria pe ani.


In [21]:
terr = "Macroregiuni, regiuni de dezvoltare si judete"
cluj = df[(df[terr] == "Cluj") & (df["Sexe"] == "Total")]
cluj = cluj.sort_values("Ani_an")
cluj[[terr, "Ani_an", "Valoare"]].tail(10)

,"Macroregiuni, regiuni de dezvoltare si judete",Ani_an,Valoare
305,Cluj,2015,464.4
306,Cluj,2016,471.0
307,Cluj,2017,468.8
308,Cluj,2018,464.9
309,Cluj,2019,465.5
310,Cluj,2020,467.8
311,Cluj,2021,472.0
312,Cluj,2022,437.1
313,Cluj,2023,443.7
314,Cluj,2024,447.2


Fara grafice aici, intentionat: acest notebook nu adauga nicio dependinta
peste ce ii trebuie deja lui pytempo. Graficele, hartile si modelarea sunt
munca obisnuita pe un DataFrame obisnuit.


In [22]:
cluj["Valoare"].describe()

count     35.000000
mean     456.131429
std       10.760597
min      434.300000
25%      447.450000
50%      459.500000
75%      464.650000
max      472.000000
Name: Valoare, dtype: float64

## Unde afli mai mult

* `t.help()` afiseaza ghidul complet de navigare.
* `m.help()` face acelasi lucru pentru un indicator.
* `m.how()` iti da comenzile de descarcare pentru acel indicator anume.
* README-ul acopera nivelele, rolurile, forma datelor si registrul intern de
  scheme.

Un gand de incheiere. pytempo face o singura treaba: scoate datele din TEMPO,
corect si reproductibil. Analiza, vizualizarea si cartografierea nu sunt
treaba lui, si sta deoparte intentionat, ca sa poti folosi uneltele obisnuite.
